# E050 — RASTA Filtering for Audio

**Motivation:** RASTA (RelAtive SpecTrAl) filtering removes slow channel variations and convolutional noise. Classic technique in speech recognition. Testing on top of E042 (tied cov + speed TTA).

**Hypothesis:** RASTA filtering will improve robustness and reduce EER.

**Configs:**
- `no_rasta`: E042 baseline (no RASTA)
- `rasta`: RASTA bandpass filter (1-12 Hz)

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from scipy.signal import lfilter
from sklearn.mixture import GaussianMixture
import copy

from data.splits import load_manifest, iter_folds
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
print(f'{len(manifest)} samples')

E042_REF = {'mean_eer': 0.46, 'std_eer': 0.65}

In [ ]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _apply_rasta(feat, low_cut=1.0, high_cut=12.0, fs=100.0):
    """
    Apply RASTA filtering to each cepstral coefficient.
    
    Args:
        feat: T x D feature matrix
        low_cut: Low cutoff frequency (Hz)
        high_cut: High cutoff frequency (Hz)
        fs: Frame rate (Hz), typically ~100 for 10ms frames
    
    Returns:
        RASTA-filtered features
    """
    # RASTA bandpass filter
    # Standard RASTA: 4th order IIR bandpass
    # Transfer function from Hermansky & Morgan 1994
    
    # Normalized frequencies
    low = low_cut / (fs / 2)
    high = high_cut / (fs / 2)
    
    # Simple first-order bandpass approximation
    # Highpass: remove slow trends
    # Lowpass: smooth rapid fluctuations
    
    filtered = feat.copy()
    
    # Apply to each coefficient (not deltas/delta-deltas)
    for d in range(13):  # Static coefficients only
        x = feat[:, d]
        
        # Highpass filter (remove trends < 1 Hz)
        # y[n] = x[n] - x[n-1] + 0.9*y[n-1]
        b_hp = [1, -1]
        a_hp = [1, -0.9]
        y_hp = lfilter(b_hp, a_hp, x)
        
        # Lowpass filter (smooth > 12 Hz)
        # y[n] = 0.1*x[n] + 0.9*y[n-1]
        b_lp = [0.1]
        a_lp = [1, -0.9]
        y_lp = lfilter(b_lp, a_lp, y_hp)
        
        filtered[:, d] = y_lp
    
    return filtered

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def _train_ubm(X, cov_type='tied'):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type=cov_type,
                           max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

def train_rasta_model(train_df, data_dir, seed, use_rasta=False):
    rng = np.random.default_rng(seed)
    X_target, X_nontarget = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:
            wavs.append(_aug_pitch(y_wav, sr, rng))
        for y_aug in wavs:
            feat = _extract_lpcc(y_aug, sr)
            if use_rasta:
                feat = _apply_rasta(feat)
            if row["label"] == 1:
                X_target.append(feat)
            else:
                X_nontarget.append(feat)
    ubm = _train_ubm(np.vstack(X_nontarget), cov_type='tied')
    adapted = _map_adapt(ubm, np.vstack(X_target))
    return ubm, adapted

def score_with_speed_tta(ubm, adapted, test_row, data_dir, use_rasta=False):
    y_wav, sr = librosa.load(_find_wav(test_row["stem"], data_dir), sr=None, mono=True)
    scores = []
    for speed in [0.9, 1.0, 1.1]:
        if speed == 1.0:
            y_proc = y_wav
        else:
            y_proc = librosa.effects.time_stretch(y_wav, rate=speed)
        feat = _extract_lpcc(y_proc, sr)
        if use_rasta:
            feat = _apply_rasta(feat)
        llr = adapted.score(feat) - ubm.score(feat)
        scores.append(float(llr) if np.isscalar(llr) else llr.mean())
    return np.mean(scores)

print('Functions defined')

In [ ]:
configs = {'no_rasta (E042)': False, 'rasta': True}
results = []

for config_name, use_rasta in configs.items():
    print(f'\n{"="*60}')
    print(f'Testing {config_name}')
    print(f'{"="*60}')
    
    fold_eers = []
    
    for fold_idx, tr_idx, te_idx in iter_folds(manifest, n_splits=3, seed=SEED):
        print(f'  Fold {fold_idx}...', end=' ', flush=True)
        
        train_df = manifest.iloc[tr_idx]
        test_df = manifest.iloc[te_idx]
        
        ubm, adapted = train_rasta_model(train_df, DATA, SEED + fold_idx, use_rasta=use_rasta)
        
        scores, labels = [], []
        for _, row in test_df.iterrows():
            score = score_with_speed_tta(ubm, adapted, row, DATA, use_rasta=use_rasta)
            scores.append(score)
            labels.append(row['label'])
        
        eer, _ = compute_eer(np.array(labels), np.array(scores))
        fold_eers.append(eer * 100)
        print(f'{eer*100:.2f}%')
    
    mean_eer = np.mean(fold_eers)
    std_eer = np.std(fold_eers)
    print(f'{config_name}: {mean_eer:.2f} ± {std_eer:.2f}%')
    
    results.append({
        'config': config_name,
        'eer_mean': mean_eer,
        'eer_std': std_eer,
        'fold_eers': fold_eers
    })

# Summary
print('\n' + '='*70)
print('E050 RASTA FILTERING SUMMARY')
print('='*70)

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df[['config', 'eer_mean', 'eer_std']].to_string(index=False))

best_idx = results_df['eer_mean'].argmin()
best = results_df.iloc[best_idx]
print(f'\n✓ Best: {best["config"]} → {best["eer_mean"]:.2f} ± {best["eer_std"]:.2f}%')

improvement = E042_REF['mean_eer'] - best['eer_mean']
print(f'Improvement vs E042 (0.46%): {improvement:+.2f}pp')

if improvement > 0.1:
    print(f'\n✓✓ NEW AUDIO FLAGSHIP!')
elif improvement > 0:
    print(f'\n✓ Marginal improvement')
else:
    print(f'\n✗ E042 remains flagship')